# FW CADIS Local Weight Windows

Creating and utilizing local weight windows to accelerate a deep shielding simulation for a single detector

The previous example used FW-CADIS to make global weight windows which try to get good statistics everywhere in the model. Often we only care about one tally, for example the flux in a single detector. In that case we don't want to spend compute time getting particles to parts of the model we are not interested in.

This example simulates a shielded room with a neutron source in the center of the room and a detector outside one of the walls. The wall next to the detector is 100cm thick and the other three walls are 150cm thick.

OpenMC can make local weight windows with FW-CADIS by setting the ```targets``` attribute on the ```WeightWindowGenerator```. In this tutorial we make both global and local weight windows and compare them. The global weight windows are just made for the comparison, you don't need them to make the local weight windows.

The variance reduction method used for this simulation is well documented in the OpenMC documentation
https://docs.openmc.org/en/latest/usersguide/variance_reduction.html

In [ ]:
import copy
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

import openmc

# Setting the cross section path to the correct location in the docker image.
# If you are running this outside the docker image you will have to change this path to your local cross section path.
openmc.config['cross_sections'] = Path.home() / 'nuclear_data' / 'cross_sections.xml'

We create a couple of materials for the simulation

In [ ]:
mat_air = openmc.Material(name='air')
mat_air.add_element("N", 0.784431)
mat_air.add_element("O", 0.210748)
mat_air.add_element("Ar", 0.0046)
mat_air.set_density("g/cc", 0.001205)

mat_concrete = openmc.Material(name='concrete')
mat_concrete.add_element("H", 0.168759)
mat_concrete.add_element("C", 0.001416)
mat_concrete.add_element("O", 0.562524)
mat_concrete.add_element("Na", 0.011838)
mat_concrete.add_element("Mg", 0.0014)
mat_concrete.add_element("Al", 0.021354)
mat_concrete.add_element("Si", 0.204115)
mat_concrete.add_element("K", 0.005656)
mat_concrete.add_element("Ca", 0.018674)
mat_concrete.add_element("Fe", 0.00426)
mat_concrete.set_density("g/cm3", 2.3)

materials = openmc.Materials([mat_air, mat_concrete])

Now we define and plot the geometry. This is a room of air inside a concrete shield which is 100cm thick on the right hand side and 150cm thick on the other three sides. There is a small detector cell outside the right hand wall. The top and bottom surfaces are reflective which keeps the problem two dimensional and quick to run.

In [ ]:
room = openmc.model.RectangularPrism(width=100, height=100)
shield = openmc.model.RectangularPrism(width=350, height=400, origin=(-25, 0))
world = openmc.model.RectangularPrism(width=450, height=500, origin=(-25, 0), boundary_type='vacuum')
detector = openmc.model.RectangularPrism(width=20, height=20, origin=(180, 0))
bottom = openmc.ZPlane(z0=-50, boundary_type='reflective')
top = openmc.ZPlane(z0=50, boundary_type='reflective')

room_cell = openmc.Cell(region=-room & +bottom & -top, fill=mat_air, name='room')
shield_cell = openmc.Cell(region=+room & -shield & +bottom & -top, fill=mat_concrete, name='shield')
detector_cell = openmc.Cell(region=-detector & +bottom & -top, fill=mat_air, name='detector')
outside_cell = openmc.Cell(region=+shield & -world & +detector & +bottom & -top, fill=mat_air, name='outside')

geometry = openmc.Geometry([room_cell, shield_cell, detector_cell, outside_cell])
geometry.plot(basis='xy', color_by='cell', outline=True)

Next we create a 14MeV point source in the center of the room and make the settings for our continuous energy MC solver

In [ ]:
source = openmc.IndependentSource(
    space=openmc.stats.Point((0, 0, 0)),
    angle=openmc.stats.Isotropic(),
    energy=openmc.stats.Discrete([14.06e6], [1.0]),
)

settings = openmc.Settings()
settings.run_mode = 'fixed source'
settings.source = source
settings.particles = 80
settings.batches = 100
settings.inactive = 50  # random ray needs inactive batches

ce_model = openmc.Model(geometry, materials, settings)

Creates the model and plots it to check the location of the source is correct. Plotting the model instead of the geometry shows the source location in red.

In [ ]:
ce_model.plot(n_samples=1, basis='xy', color_by='cell', outline=True, source_kwargs={'color': 'red'})

Just like the previous example we make a copy of the model and convert it to multigroup and random ray. Random ray is needed for the FW-CADIS algorithm.

In [ ]:
rr_model = copy.deepcopy(ce_model)
rr_model.convert_to_multigroup(method='stochastic_slab', overwrite_mgxs_library=True, nparticles=2000)
rr_model.convert_to_random_ray()

Now we setup a mesh that is used for the weight windows and for subdividing the random ray source regions. We also make two tallies, a mesh flux tally for viewing the statistics across the whole model and a detector flux tally which is the tally we actually care about.

In [ ]:
mesh = openmc.RegularMesh().from_domain(geometry)
mesh.dimension = (45, 50, 1)

mesh_tally = openmc.Tally(name='mesh flux')
mesh_tally.filters = [openmc.MeshFilter(mesh)]
mesh_tally.scores = ['flux']

detector_tally = openmc.Tally(name='detector flux')
detector_tally.filters = [openmc.CellFilter(detector_cell)]
detector_tally.scores = ['flux']

tallies = openmc.Tallies([mesh_tally, detector_tally])

# the target tallies must be present on the random ray model
rr_model.tallies = tallies
rr_model.settings.random_ray['source_region_meshes'] = [(mesh, [rr_model.geometry.root_universe])]

First we generate the global weight windows, this is the same as the previous example. The weight windows get saved to weight_windows.h5 so we rename the file to avoid overwriting it in the next step.

In [ ]:
wwg = openmc.WeightWindowGenerator(method='fw_cadis', mesh=mesh)
rr_model.settings.weight_window_generators = wwg

rr_model.run()
Path('weight_windows.h5').rename('weight_windows_global.h5')

Now we generate the local weight windows. The only change needed is to set the ```targets``` attribute to the tally or tallies that we want to optimise for.

In [ ]:
wwg.targets = openmc.Tallies([detector_tally])

rr_model.run()
Path('weight_windows.h5').rename('weight_windows_local.h5')

Now we plot both sets of weight windows with the geometry overlay. Low values (dark colors) are where particles get split.

We can see the global weight windows drop off through all four walls of the shield. The local weight windows only drop off through the wall next to the detector and they increase in the other three walls, so neutrons heading away from the detector get killed off instead of split.

In [ ]:
weight_windows = {
    'global': openmc.WeightWindowsList.from_hdf5('weight_windows_global.h5'),
    'local': openmc.WeightWindowsList.from_hdf5('weight_windows_local.h5'),
}
mesh_extent = mesh.bounding_box.extent['xy']

fig, axes = plt.subplots(ncols=2, figsize=(12, 5.5))
for ax, (name, wws) in zip(axes, weight_windows.items()):
    im = ax.imshow(wws[0].lower_ww_bounds.squeeze().T, origin='lower', extent=mesh_extent, norm=LogNorm())
    plt.colorbar(im, ax=ax)
    ce_model.plot(outline='only', extent=mesh_extent, axes=ax, color_by='cell')
    ax.set_title(f'{name} lower weight window bounds')

Next we run the continuous energy simulation twice, once with each set of weight windows and with the same number of particles.

For each simulation we print the detector flux, the relative error and the figure of merit (FOM). The FOM is 1 / (relative error² x simulation time) so a higher FOM means a more efficient simulation. We also plot the relative error of the mesh flux tally.

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(12, 5.5))

for ax, (name, wws) in zip(axes, weight_windows.items()):
    settings = openmc.Settings()
    settings.run_mode = 'fixed source'
    settings.source = source
    settings.particles = 2000
    settings.batches = 20
    settings.weight_windows = wws
    settings.weight_window_checkpoints = {'collision': True, 'surface': True}

    model = openmc.Model(geometry, materials, settings, tallies)
    sp_filename = model.run(output=False)

    with openmc.StatePoint(sp_filename) as sp:
        detector_result = sp.get_tally(name='detector flux')
        mesh_result = sp.get_tally(name='mesh flux')
        time = sp.runtime['simulation']

    rel_err = detector_result.std_dev.item() / detector_result.mean.item()
    fom = 1 / (rel_err**2 * time)
    print(f'{name}: detector flux={detector_result.mean.item():.3e} rel. error={rel_err:.3f} time={time:.1f}s FOM={fom:.1f}')

    mesh_rel_err = mesh_result.get_reshaped_data(value='rel_err', expand_dims=True).squeeze()
    im = ax.imshow(mesh_rel_err.T, origin='lower', extent=mesh_extent, norm=LogNorm(vmin=0.01, vmax=1))
    plt.colorbar(im, ax=ax)
    ce_model.plot(outline='only', extent=mesh_extent, axes=ax, color_by='cell')
    ax.set_title(f'{name} flux relative error')

Comparing the two simulations we notice that the global weight windows give a similar relative error everywhere in the model, even behind the three thick walls. The local weight windows give a lower relative error at the detector and the simulation is quicker. The downside is the statistics behind the other walls are terrible and the white voxels didn't get any scores at all, but we are not interested in those regions.

On this computer the FOM was 2 for the global weight windows and 27 for the local weight windows, so the local weight windows were over 10 times more efficient for the detector tally. Your numbers will differ a bit as the relative error is only a rough estimate with this few particles.

The two detector flux values are close to each other and if you increase the number of particles they converge on the same value.

If you have time there are a few things you could try to extend this task.

The ```targets``` attribute accepts more than one tally, so the weight windows can be optimised for several detectors at once.

- Try adding a second detector cell outside one of the 150cm thick walls with its own flux tally
- Add both detector tallies to ```wwg.targets``` and make the local weight windows again
- Plot the new weight windows, which walls do they work on now?
- Compare the relative error of the two detector tallies. FW-CADIS weights each target by its forward flux so the detector behind the thick wall should not get neglected.
- How does the FOM compare with the global weight windows now? What would you expect if there was a detector outside all four walls?